In [1]:
import torch
import torch.nn as nn
import torch.utils.model_zoo as model_zoo
import torch
from torch.nn import functional as F
from torch.utils.data import Dataset
import cv2
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
from torch import optim
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from torchvision import datasets, transforms, models
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import glob
from tqdm import tqdm


from sklearn.model_selection import train_test_split, KFold

In [2]:
test_transform224 = A.Compose([
    A.SmallestMaxSize(224),
    A.CenterCrop(224, 224),
    A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            max_pixel_value=255.0),
    ToTensorV2()
])


In [3]:
def vgg19():
    """VGG 19-layer model (configuration "E")
        model pre-trained on ImageNet
    """
    model = VGG(make_layers(cfg['E']))
    # model.load_state_dict(model_zoo.load_url(model_urls['vgg19']), strict=False) --> pretrain introduction
    return model
# ————————————————
# 版权声明：本文为CSDN博主「Tra_cy」的原创文章，遵循CC 4.0 BY-SA版权协议，转载请附上原文出处链接及本声明。
# 原文链接：https://blog.csdn.net/Tra_cy/article/details/103269899
class VGG(nn.Module):
#nn.Module是一个特殊的nn模块，加载nn.Module，这是为了继承父类
    def __init__(self, features):
        super(VGG, self).__init__()
        #super 加载父类中的__init__()函数
        self.features = features
        
       

        self.reg_layer = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 1, 1)
        )
        self.flatten_layer = nn.Flatten()
        self.dnn = nn.Sequential(nn.Linear(14*14,16),
                                 
                                 nn.Dropout(0.2),
                                #  nn.Linear(64,32),
                                 
                                #  nn.Dropout(0.1),
                                #  nn.Linear(32,16),
                                 
                                 nn.Linear(16,1),)
                                 
        # self.fully_connected_layer = nn.Linear(in_features = 14*14, out_features = 1)



    def forward(self, x):
      
        x = self.features(x)
        # print(x)
        
        
        # x = F.upsample_bilinear(x, scale_factor=1/14)

        x = self.reg_layer(x)
        # print(x)
        x = self.flatten_layer(x)
        # print(x)
        # x = self.fully_connected_layer(x)
        x = self.dnn(x)
 
        return torch.abs(x)

def make_layers(cfg, batch_norm=True):
    layers = []
    in_channels = 3
    for v in cfg:
        if v == 'M':
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
        else:
            conv2d = nn.Conv2d(in_channels, v, kernel_size=3, padding=1)
            if batch_norm:
                layers += [conv2d, nn.BatchNorm2d(v), nn.ReLU(inplace=True)]
            else:
                layers += [conv2d, nn.ReLU(inplace=True)]
            in_channels = v
    
    return nn.Sequential(*layers)
# ————————————————
# 版权声明：本文为CSDN博主「Tra_cy」的原创文章，遵循CC 4.0 BY-SA版权协议，转载请附上原文出处链接及本声明。
# 原文链接：https://blog.csdn.net/Tra_cy/article/details/103269899

In [4]:
class pawpularity_Dataset_test(Dataset):
    def __init__(self, image_path_pic  ,transform=False):
        self.image_path_pic = image_path_pic
        
        self.transform = transform
        
    def __len__(self):
        return len(self.image_path_pic)

    def __getitem__(self, idx):
       
        image_filepath = self.image_path_pic[idx]
        
        image = cv2.imread(image_filepath)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(np.float64)
    

        # check: if ID match or not
        ID_pic = str(self.image_path_pic[idx]).split('/')[-1]
        ID = str(self.image_path_pic[idx]).split('/')[-1].split('.')[0]

    
        
        if self.transform is not None:
            image = self.transform(image=image)["image"]
        
        # label = torch.from_numpy(np.array(float(label))).float()
        
        
        # return image, torch.from_numpy(np.array(label)).float()
        return image , ID

In [5]:
filenames_pic_test = glob.glob("/kaggle/input/petfinder-pawpularity-score/test/*.jpg")
# filenames_pic_test = glob.glob("/homes/pinjchen/Machine_learning_course/pawpularity/petfinder-pawpularity-score/test/*jpg")

test_dataset = pawpularity_Dataset_test(filenames_pic_test,transform = test_transform224)
test_loader = DataLoader(
    test_dataset, batch_size=1, shuffle=True
)

In [6]:
model = torch.load('/kaggle/input/check-point-ep2/model_0525_1epoch_for_abs.pth').cpu()
model.eval()
ID = []
pawpularity = []
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with torch.no_grad():
        for i , data in enumerate(test_loader):
                
                test_feat = data[:][0].float()
                z = torch.zeros(1, 3,224,224).float()
 
                prediction_pawpularity = model(test_feat)
         
                ID.append(str(data[1]).replace(')', '').replace('(', '').replace(',', '')[1:-1])

                                                                                                  
                pawpularity.append(((float(prediction_pawpularity))))
                # print(data[1],prediction_pawpularity)   

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/check-point-ep2/model_0525_1epoch_for_abs.pth'

In [7]:
data = {'Id':ID
        ,'Pawpularity':pawpularity}
data = pd.DataFrame(data)
data = data.sort_values(by=['Id'])
data


NameError: name 'ID' is not defined

In [8]:
data.to_csv(f'submission.csv', index=0)


NameError: name 'data' is not defined